# Tema 6 — Data Augmentation (PLN I)
## Ejercicio 1 (rotten_tomatoes): TF‑IDF + SVM + Aumento de datos

Este notebook resuelve el **Ejercicio 1** propuesto en la hoja de ejercicios del Tema 6:

1. Cargar un subconjunto del dataset **`rotten_tomatoes`** (comentarios de películas, **positivo/negativo**).
2. Sacar estadísticas (número de textos por clase).
3. Hacer **split** train/test y **desbalancear** el conjunto de entrenamiento.
4. Entrenar un baseline clásico: **TF‑IDF + SVM**.
5. Aplicar **Data Augmentation (DA)** **solo sobre la clase minoritaria del train**, reentrenar y comparar resultados.
6. Probar **varios métodos de DA** usando librerías vistas en clase (**nlpaug**, **TextAttack**) y, opcionalmente, **TextAugment**.

> Nota importante:
> - El DA se aplica **antes del entrenamiento** y **nunca** sobre validación/test.
> - No todos los DA mejoran resultados: depende de la tarea, del modelo y de la calidad del texto aumentado.


In [16]:
# ==========================
# 0) Instalación (si hace falta)
# ==========================
# Ejecuta esta celda SOLO si te faltan librerías.
# En entornos tipo Google Colab suele funcionar sin problemas.

# !pip -q install datasets scikit-learn pandas numpy matplotlib
# !pip -q install nlpaug nltk
# !pip -q install textattack
# (Opcional) TextAugment:
# !pip -q install textaugment

# Descargas de NLTK (WordNet y recursos básicos)
import nltk
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")
try:
    nltk.data.find("corpora/omw-1.4")
except LookupError:
    nltk.download("omw-1.4")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hugo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hugo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [17]:
# ==========================
# 1) Imports y semillas
# ==========================
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. Carga del dataset y estadísticas

Usaremos `rotten_tomatoes`, con etiquetas binarias:
- `0` → negativo
- `1` → positivo

El ejercicio pide **subconjunto**: tomaremos una muestra aleatoria (para que el entrenamiento sea rápido) y haremos un **split train/test** estratificado.


In [18]:
# ==========================
# 1) Carga y muestreo
# ==========================
ds = load_dataset("rotten_tomatoes")

# Unificamos (train + validation + test) para poder samplear y luego re-splitear
texts = []
labels = []
for split in ["train", "validation", "test"]:
    texts.extend(ds[split]["text"])
    labels.extend(ds[split]["label"])

data = pd.DataFrame({"text": texts, "label": labels})

# Subconjunto (ajusta N si quieres más/menos)
N = 4000
if N < len(data):
    data = data.sample(n=N, random_state=RANDOM_SEED).reset_index(drop=True)

# Estadísticas por clase
class_counts = data["label"].value_counts().sort_index()
display(class_counts.rename(index={0:"neg", 1:"pos"}).to_frame("count"))

print(f"Total ejemplos usados: {len(data)}")


,count
label,
neg,1977
pos,2023


Total ejemplos usados: 4000


## 2. Split train/test (estratificado)

- **Estratificado** significa que train y test mantienen (aproximadamente) la misma proporción de clases.
- Separaremos también un pequeño **validation** desde train para poder comparar de forma más estable si se quiere (opcional).


In [19]:
# ==========================
# 2) Split train/test
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    data["text"].values,
    data["label"].values,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=data["label"].values
)

# (Opcional) split de validación dentro de train
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,
    random_state=RANDOM_SEED,
    stratify=y_train
)

def show_counts(name, y):
    vc = pd.Series(y).value_counts().sort_index()
    print(f"--- {name} ---")
    print(pd.DataFrame({"count": vc, "ratio": (vc / vc.sum()).round(3)}).rename(index={0:"neg", 1:"pos"}))
    print()

show_counts("Train (antes de desbalanceo)", y_train)
show_counts("Val", y_val)
show_counts("Test", y_test)


TypeError: only integer scalar arrays can be converted to a scalar index

## 3. Desbalanceo del conjunto de entrenamiento

La idea es simular un escenario típico donde hay **pocos ejemplos** de una clase (la **minoritaria**).

Implementación:
- Elegimos una clase para hacerla minoritaria (por defecto, la **positiva**).
- Nos quedamos con un porcentaje pequeño de esa clase en train.
- **Validación y test NO se tocan** (siguen representando el problema real).


In [ ]:
# ==========================
# 3) Desbalanceo artificial del train
# ==========================
minority_label = 1  # 1=positivo (puedes cambiarlo a 0 si quieres)
keep_ratio_minority = 0.2  # nos quedamos con el 20% de la clase minoritaria

idx_min = np.where(y_train == minority_label)[0]
idx_maj = np.where(y_train != minority_label)[0]

rng = np.random.default_rng(RANDOM_SEED)
keep_min = rng.choice(idx_min, size=max(1, int(len(idx_min) * keep_ratio_minority)), replace=False)

idx_new = np.concatenate([idx_maj, keep_min])
rng.shuffle(idx_new)

X_train_imb = X_train[idx_new]
y_train_imb = y_train[idx_new]

show_counts("Train (DESBALANCEADO)", y_train_imb)


## 4. Baseline clásico: TF‑IDF + SVM (LinearSVC)

### Conceptos clave
- **TF‑IDF**: representa cada texto como un vector donde cada dimensión es una palabra (o n-grama).  
  - TF (Term Frequency): frecuencia en el documento.
  - IDF (Inverse Document Frequency): penaliza palabras muy frecuentes en el corpus.
- **SVM** (Support Vector Machine): clasificador lineal fuerte para texto, especialmente con TF‑IDF.

Entrenaremos con el train desbalanceado y evaluaremos en test.


In [ ]:
# ==========================
# 4) Baseline TF-IDF + LinearSVC
# ==========================
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",      # como rotten_tomatoes está en inglés
    ngram_range=(1, 2),
    max_features=50000
)

Xtr_vec = vectorizer.fit_transform(X_train_imb)
Xte_vec = vectorizer.transform(X_test)

svm = LinearSVC(random_state=RANDOM_SEED)
svm.fit(Xtr_vec, y_train_imb)

pred = svm.predict(Xte_vec)

print("Baseline (train desbalanceado)")
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("F1 macro:", round(f1_score(y_test, pred, average="macro"), 4))
print("\nClassification report:\n", classification_report(y_test, pred, digits=4))

cm = confusion_matrix(y_test, pred)
print("Confusion matrix:\n", cm)


## 5. Data Augmentation (DA) sobre la clase minoritaria

Probaremos métodos típicos de los apuntes:
- **EDA (Easy Data Augmentation)**: operaciones simples a nivel de palabra (sinónimos, inserción, swap, borrado).
- **NLP Aug**: DA a nivel carácter/palabra/oración (Keyboard/OCR, WordNet, etc.).
- **TextAttack**: framework con recetas de augmentación (p. ej., WordNetAugmenter).

### Estrategia de oversampling con DA
1. Identificamos la clase minoritaria en el train desbalanceado.
2. Generamos textos aumentados **solo de esa clase** hasta equilibrar (o acercarnos) a la clase mayoritaria.
3. Reentrenamos el pipeline TF‑IDF + SVM y comparamos métricas.

> Importante: DA puede degradar semántica o coherencia si se usa demasiado agresivo.


In [ ]:
# ==========================
# 5) Funciones auxiliares: construir train aumentado
# ==========================
def make_balanced_with_augmentation(X_train, y_train, augment_fn, target_ratio=1.0, max_new_per_sample=3):
    '''
    Crea un nuevo train donde se aumenta la clase minoritaria con augment_fn hasta aproximar balance.

    - augment_fn(text) -> list[str] o str: genera variantes del texto.
    - target_ratio: ratio deseado minoritaria/mayoritaria (1.0 = balance perfecto).
    - max_new_per_sample: límite de aumentos por ejemplo para evitar explosión de datos.
    '''
    X_train = np.array(X_train, dtype=object)
    y_train = np.array(y_train)

    counts = pd.Series(y_train).value_counts()
    maj_label = counts.idxmax()
    min_label = counts.idxmin()
    n_maj = counts[maj_label]
    target_min = int(n_maj * target_ratio)

    X_new = list(X_train)
    y_new = list(y_train)

    if sum(y_train == min_label) >= target_min:
        return np.array(X_new, dtype=object), np.array(y_new)

    min_idx = np.where(y_train == min_label)[0]
    rng = np.random.default_rng(RANDOM_SEED)

    while sum(np.array(y_new) == min_label) < target_min:
        i = int(rng.choice(min_idx, size=1)[0])
        base_text = X_train[i]

        augmented = augment_fn(base_text)
        if isinstance(augmented, str):
            augmented = [augmented]
        augmented = [t for t in augmented if isinstance(t, str) and t.strip()]
        augmented = augmented[:max_new_per_sample]

        for t in augmented:
            X_new.append(t)
            y_new.append(min_label)
            if sum(np.array(y_new) == min_label) >= target_min:
                break

    return np.array(X_new, dtype=object), np.array(y_new)


## 5.1 Aumentación con **nlpaug** (WordNet + EDA)

Probamos tres tipos típicos:
1. **Synonym replacement (WordNet)**.
2. **Random deletion**.
3. **Random swap**.

Se usan parámetros suaves (pocos cambios) para no romper la semántica.


In [ ]:
# ==========================
# 5.1) NLP Aug: Word-level augmenters (EDA-like)
# ==========================
def build_nlpaug_augmenters():
    import nlpaug.augmenter.word as naw
    aug_syn = naw.SynonymAug(aug_src="wordnet")
    aug_del = naw.RandomWordAug(action="delete", aug_p=0.1)
    aug_swap = naw.RandomWordAug(action="swap", aug_p=0.1)
    return aug_syn, aug_del, aug_swap

try:
    aug_syn, aug_del, aug_swap = build_nlpaug_augmenters()
    HAVE_NLPAUG = True
    print("nlpaug OK")
except Exception as e:
    HAVE_NLPAUG = False
    print("No se pudo importar/usar nlpaug. Error:", e)

def augment_nlpaug_combo(text, n=2):
    if not HAVE_NLPAUG:
        raise RuntimeError("nlpaug no disponible")
    out = []
    for _ in range(n):
        t = text
        t = aug_syn.augment(t)
        t = aug_del.augment(t)
        t = aug_swap.augment(t)
        out.append(t)
    return out

demo_text = "This movie was surprisingly good, with great acting and a solid plot."
if HAVE_NLPAUG:
    print("Original:", demo_text)
    print("Augmented:", augment_nlpaug_combo(demo_text, n=2))


In [ ]:
# ==========================
# Entrenar/evaluar tras DA con nlpaug
# ==========================
if HAVE_NLPAUG:
    X_train_aug, y_train_aug = make_balanced_with_augmentation(
        X_train_imb, y_train_imb,
        augment_fn=lambda t: augment_nlpaug_combo(t, n=2),
        target_ratio=1.0,
        max_new_per_sample=2
    )

    show_counts("Train tras DA (nlpaug)", y_train_aug)

    vec = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=50000
    )
    Xtr = vec.fit_transform(X_train_aug)
    Xte = vec.transform(X_test)

    clf = LinearSVC(random_state=RANDOM_SEED)
    clf.fit(Xtr, y_train_aug)
    pred_aug = clf.predict(Xte)

    print("DA con nlpaug (WordNet+delete+swap)")
    print("Accuracy:", round(accuracy_score(y_test, pred_aug), 4))
    print("F1 macro:", round(f1_score(y_test, pred_aug, average='macro'), 4))
    print("\nClassification report:\n", classification_report(y_test, pred_aug, digits=4))
else:
    print("Salto: nlpaug no disponible.")


## 5.2 Aumentación a nivel de carácter (Keyboard/OCR) con **nlpaug** (opcional)

- **Keyboard augmenter**: simula errores de teclas cercanas.
- **OCR augmenter**: simula confusiones de grafía.

Úsalo con parámetros muy suaves, porque mete ruido.


In [ ]:
# ==========================
# 5.2) NLP Aug: Char-level augmenters (opcionales)
# ==========================
def build_char_augmenters():
    import nlpaug.augmenter.char as nac
    aug_kb = nac.KeyboardAug(aug_char_p=0.02)
    aug_ocr = nac.OcrAug(aug_char_p=0.02)
    return aug_kb, aug_ocr

try:
    aug_kb, aug_ocr = build_char_augmenters()
    HAVE_CHAR_AUG = True
    print("Char augmenters OK")
except Exception as e:
    HAVE_CHAR_AUG = False
    print("No se pudo usar char augmenters. Error:", e)

def augment_char_noise(text, n=2):
    if not HAVE_CHAR_AUG:
        raise RuntimeError("Char augmenters no disponibles")
    out = []
    for _ in range(n):
        t = text
        t = aug_kb.augment(t)
        t = aug_ocr.augment(t)
        out.append(t)
    return out

if HAVE_CHAR_AUG:
    print("Original:", demo_text)
    print("Augmented:", augment_char_noise(demo_text, n=2))


## 5.3 Aumentación con **TextAttack** (WordNet)

TextAttack incluye `WordNetAugmenter`, que genera variantes usando sinónimos.


In [ ]:
# ==========================
# 5.3) TextAttack: WordNetAugmenter
# ==========================
try:
    from textattack.augmentation import WordNetAugmenter
    HAVE_TEXTATTACK = True
    ta_aug = WordNetAugmenter()
    print("TextAttack OK")
except Exception as e:
    HAVE_TEXTATTACK = False
    print("No se pudo importar TextAttack. Error:", e)

def augment_textattack_wordnet(text, n=2):
    if not HAVE_TEXTATTACK:
        raise RuntimeError("TextAttack no disponible")
    out = []
    for _ in range(n):
        augmented = ta_aug.augment(text)
        if isinstance(augmented, str):
            augmented = [augmented]
        out.extend(augmented[:1])
    return out[:n]

if HAVE_TEXTATTACK:
    print("Original:", demo_text)
    print("Augmented:", augment_textattack_wordnet(demo_text, n=2))


In [ ]:
# ==========================
# Entrenar/evaluar tras DA con TextAttack
# ==========================
if HAVE_TEXTATTACK:
    X_train_aug2, y_train_aug2 = make_balanced_with_augmentation(
        X_train_imb, y_train_imb,
        augment_fn=lambda t: augment_textattack_wordnet(t, n=2),
        target_ratio=1.0,
        max_new_per_sample=2
    )

    show_counts("Train tras DA (TextAttack)", y_train_aug2)

    vec2 = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=50000
    )
    Xtr2 = vec2.fit_transform(X_train_aug2)
    Xte2 = vec2.transform(X_test)

    clf2 = LinearSVC(random_state=RANDOM_SEED)
    clf2.fit(Xtr2, y_train_aug2)
    pred_aug2 = clf2.predict(Xte2)

    print("DA con TextAttack (WordNetAugmenter)")
    print("Accuracy:", round(accuracy_score(y_test, pred_aug2), 4))
    print("F1 macro:", round(f1_score(y_test, pred_aug2, average='macro'), 4))
    print("\nClassification report:\n", classification_report(y_test, pred_aug2, digits=4))
else:
    print("Salto: TextAttack no disponible.")


## 5.4 (Opcional) DA avanzado: Back Translation y/o Contextual Embeddings

Estos métodos suelen ser más costosos (descarga de modelos, GPU), pero generan textos más naturales.
Se dejan celdas orientativas.


In [ ]:
# ==========================
# 5.4) Opcional: Back Translation con nlpaug (puede tardar y descargar modelos)
# ==========================
# !pip -q install transformers sentencepiece
# import nlpaug.augmenter.word as naw
# back_trans = naw.BackTranslationAug(
#     from_model_name="Helsinki-NLP/opus-mt-en-de",
#     to_model_name="Helsinki-NLP/opus-mt-de-en"
# )
# print(back_trans.augment(demo_text))

# ==========================
# 5.4) Opcional: Contextual embeddings (BERT) con nlpaug
# ==========================
# import nlpaug.augmenter.word as naw
# bert_aug = naw.ContextualWordEmbsAug(
#     model_path="bert-base-uncased",
#     action="substitute"
# )
# print(bert_aug.augment(demo_text))


## 6. Comparación y conclusiones

Analiza:
- F1 macro (clave con desbalanceo).
- Recall de la clase minoritaria.
- Si el DA mete ruido y empeora.

Abajo se construye una tabla comparativa si se han ejecutado las secciones correspondientes.


In [ ]:
# ==========================
# 6) Tabla comparativa de métricas
# ==========================
results = []
results.append({
    "method": "baseline",
    "accuracy": accuracy_score(y_test, pred),
    "f1_macro": f1_score(y_test, pred, average="macro"),
})

if 'pred_aug' in globals():
    results.append({
        "method": "nlpaug_wordnet+delete+swap",
        "accuracy": accuracy_score(y_test, pred_aug),
        "f1_macro": f1_score(y_test, pred_aug, average="macro"),
    })

if 'pred_aug2' in globals():
    results.append({
        "method": "textattack_wordnet",
        "accuracy": accuracy_score(y_test, pred_aug2),
        "f1_macro": f1_score(y_test, pred_aug2, average="macro"),
    })

pd.DataFrame(results).sort_values("f1_macro", ascending=False)
